In [0]:
#Simple test cell
print("MLOps Capstone - Databricks Free Edition")

print("Spark version:", spark.version)

##Load the dataset

In [0]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np

dataset = load_breast_cancer(as_frame=True)

pdf = dataset.frame.copy()

# Keep the target as an integer for classification.
pdf["target"] = pdf["target"].astype("int")

print("Dataset shape:", pdf.shape)
print("Columns:", pdf.columns.tolist())
display(pdf.head())

##Explore the data

In [0]:
print("Missing values:")
display(pdf.isnull().sum().sort_values(ascending=False).head(10))

print("Target distribution:")
display(pdf["target"].value_counts().sort_index())

print("Statistical summary:")
display(pdf.describe().T)

## Data Exploration Summary

- **Rows and features:** The breast cancer dataset contains **569 rows** and **31 columns** (30 numeric features + 1 target column).
- **Missing values:** There are **no missing values** in any column, so no imputation is needed.
- **Target balance:** The target distribution is **imbalanced** — 357 benign (class 0) vs. 212 malignant (class 1). While not extreme, the skew means accuracy alone can be misleading.
- **Why metrics beyond accuracy matter:** Because the classes are imbalanced, a model could achieve high accuracy by over-predicting the majority class while missing many malignant cases. Metrics such as **precision**, **recall**, **F1-score**, and **ROC-AUC** provide a fuller picture of how well the model detects the minority (malignant) class, which is critical in a medical context where false negatives carry a high cost.

##Create a Spark DataFrame

In [0]:
sdf = spark.createDataFrame(pdf)

print("Spark row count:", sdf.count())
sdf.printSchema()

display(sdf.limit(5))

##Perform Spark-based exploration

In [0]:
from pyspark.sql import functions as F

display(
    sdf.groupBy("target")
       .count()
       .orderBy("target")
)

display(
    sdf.select(
        F.avg("mean radius").alias("avg_mean_radius"),
        F.avg("mean texture").alias("avg_mean_texture"),
        F.avg("mean area").alias("avg_mean_area")
    )
)

##Prepare training and test data

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

X = pdf.drop(columns=["target"])
y = pdf["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

##Configure MLflow

In [0]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment(
    "/Users/heinn2004@gmail.com/mlops-classification-capstone-exp"
)

##Run three experiments

In [0]:
experiments = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=1.0,
            max_iter=2000,
            random_state=42
        ))
    ]),

    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        random_state=42
    ),

    "gradient_boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=2,
        random_state=42
    )
}

results = []

for name, model in experiments.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        probabilities = model.predict_proba(X_test)[:, 1]

        accuracy = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions)
        auc = roc_auc_score(y_test, probabilities)

        mlflow.log_param("algorithm", name)
        mlflow.log_param("test_size", 0.20)
        mlflow.log_param("random_state", 42)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("roc_auc", auc)

        mlflow.sklearn.log_model(
            sk_model=model,
            name="model"
        )

        results.append({
            "model": name,
            "accuracy": accuracy,
            "f1": f1,
            "roc_auc": auc
        })

results_df = pd.DataFrame(results).sort_values(
    "roc_auc",
    ascending=False
)

display(results_df)

##Review experiment results

Three classification algorithms were evaluated using the same training and test datasets, with an 80/20 train-test split and `random_state=42`. The models were compared using accuracy, F1 score, and ROC-AUC.

### Experiment Comparison

| Model | Accuracy | F1 Score | ROC-AUC |
|---|---:|---:|---:|
| Logistic Regression | 0.982 | 0.986 | 0.995 |
| Random Forest | 0.956 | 0.966 | 0.994 |
| Gradient Boosting | 0.947 | 0.959 | 0.991 |

### Findings

**Logistic Regression achieved the best overall performance** among the three evaluated models. It obtained an accuracy of 98.2%, an F1 score of 0.986, and a ROC-AUC of 0.995.

Random Forest achieved an accuracy of 95.6%, an F1 score of 0.966, and a ROC-AUC of 0.994. Gradient Boosting achieved an accuracy of 94.7%, an F1 score of 0.959, and a ROC-AUC of 0.991.

All three models demonstrated strong classification performance on the test dataset. Logistic Regression performed best on all three reported metrics, making it the preferred candidate for the next stage of the pipeline.

### Model Selection Rationale

Logistic Regression was selected because it achieved the highest accuracy, F1 score, and ROC-AUC among the evaluated models. It also provides a relatively simple and interpretable baseline for a classification task.

The selected model will be validated before registration and deployment. Since the dataset's positive class represents benign tumors, further evaluation should pay particular attention to recall for the malignant class to understand the risk of false-negative predictions.

### MLflow Experiment Tracking

All three experiments were logged to MLflow. The tracked information includes the algorithm, random state, test-set proportion, evaluation metrics, and model artifacts. This enables experiment comparison and supports reproducibility.

The next step is to validate the selected Logistic Regression model, log it as the final candidate, and register it in Unity Catalog if the required functionality and permissions are available in Databricks Free Edition.

##Select and retrain the best model

In [0]:
best_name = results_df.iloc[0]["model"]
best_model = experiments[best_name]

print("Selected model:", best_name)
print(
    "Selected ROC-AUC:",
    results_df.iloc[0]["roc_auc"]
)

In [0]:
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
# Train the selected model
best_model.fit(X_train, y_train)

# Generate predictions for evaluation
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Create input and output examples for the model signature
input_example = X_train.head(5)
signature_predictions = best_model.predict(input_example)
signature = infer_signature(input_example, signature_predictions)

# Log the final model and its signature to MLflow
with mlflow.start_run(run_name="final_selected_model"):
    mlflow.log_param("selected_model", best_name)

    mlflow.log_metric(
        "test_accuracy",
        accuracy_score(y_test, y_pred)
    )
    mlflow.log_metric(
        "test_f1",
        f1_score(y_test, y_pred)
    )
    mlflow.log_metric(
        "test_roc_auc",
        roc_auc_score(y_test, y_prob)
    )

    model_info = mlflow.sklearn.log_model(
        sk_model=best_model,
        name="selected_model",
        signature=signature,
        input_example=input_example
    )

print("Model URI:", model_info.model_uri)
print("Model signature:", signature)

In [0]:
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name="workspace.mlops.breast_cancer_classifier"
)

print("Registered model:", registered_model.name)
print("Registered version:", registered_model.version)

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

model_name = "workspace.mlops.breast_cancer_classifier"
version = registered_model.version

client.set_registered_model_alias(
    model_name,
    "challenger",
    version
)

# Promote only after validation.
client.set_registered_model_alias(
    model_name,
    "champion",
    version
)

##Deploy and test the prediction endpoint

First, check Serving in your Databricks workspace. Free Edition has limits on serving endpoints and does not provide GPU serving endpoints, so do not assume that every custom model can be deployed. 




If your registered model is eligible for serving:
1. Open the registered model in Catalog Explorer.
2. Select the validated champion version.
3. Create a serving endpoint using the available UI.
4. Wait until the endpoint reports that it is ready.
5. Send a test request and record the response.
A sample request body for an endpoint configured for MLflow's dataframe_split input format looks like this:

In [0]:
import requests
import pandas as pd

# Set these to the actual endpoint URL and
# a securely retrieved Databricks access token.
endpoint_url = "YOUR_SERVING_ENDPOINT_URL"
token = "YOUR_DATABRICKS_TOKEN"

sample = X_test.iloc[[0]]

response = requests.post(
    endpoint_url,
    headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    },
    json={
        "dataframe_split": sample.to_dict(orient="split")
    },
    timeout=60
)

print("HTTP status:", response.status_code)
print(response.text)